In [1]:
import requests
import torch
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"


model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)



Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [36]:
all_files = []
img_dir = """/projects/matsci/vlm_microscopy/Microscopy/NFFA/original_data/"""
for path, subdirs, files in os.walk(img_dir):
    for name in files:
        all_files.append(os.path.join(path, name))

In [3]:
folder_id_map = {"Biological": 1,
                "Fibres": 2,
                "Films_Coated_Surface": 3,
                "MEMS_devices_and_electrodes": 4,
                "Nanowires": 5,
                "Particles": 6,
                "Patterned_surface": 7,
                "Porous_Sponge": 8,
                "Powder": 9,
                "Tips": 10}

class_id_map = {"biological": 1,
               "fibers": 2,
               "coated film": 3,
               "mems": 4,
               "nanowire": 5,
               "particles": 6,
               "patterned surface": 7,
               "porous sponge": 8,
               "powder": 9,
               "tips": 10}


In [4]:
len(all_files)

250

In [5]:
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "I have SEM images that may belong to any of these classes - (1) biological, (2) fibers, (3) coated films, (4) MEMS, (5) nanowires, (6) particles, (7) patterned surfaces, (8) porous sponge, (9) powder, or (10) tips. Can you please identify the image type using one of the above classes? Please answer with the class number only. I want you to make a guess even if you are not sure. I do not want any extra information, only the class number. If it is impossible to determine the class, please answer with `NaN', and nothing else."}
    ]}
]

In [6]:
all_files[0]

'/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Nanowires/b00c36659b4fd823aa7bb75bb804951f.jpg'

In [7]:
from tqdm import tqdm

responses = []
actuals = []
for img_path in tqdm(all_files, desc="Processing items"):
    image = Image.open(img_path)
    ground_truth = folder_id_map[img_path.split("/")[-2]]
    actuals.append(ground_truth)

    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    input_length = inputs.input_ids.shape[1]

    output = model.generate(**inputs, max_new_tokens=50)
    res = processor.decode(output[0][input_length:], skip_special_tokens=True)
    responses.append(res)
    # print(img_path)
    # print(res)
    

Processing items: 100%|███████████████████████████████████████████████████████████| 250/250 [01:01<00:00,  4.04it/s]


In [8]:
# for path in all_files:
#     folder = "/".join(path.split("/")[:-1])
#     file = path.split("/")[-1].split("_")[-1]
#     new_path = folder+"/"+file
#     # print(path)
#     # print(new_path)
#     os.rename(path, new_path)

In [16]:
responses

['5.',
 '1.',
 '8.',
 '7',
 '5.',
 '5.',
 '6.',
 '6',
 '6',
 '5.',
 '9.',
 '5.',
 '7.',
 '6.',
 '3.',
 '8.',
 '5.',
 '6',
 '8.',
 '3',
 '8.',
 '8.',
 '3.',
 '6',
 '6',
 '6',
 '8.',
 '3.',
 '9',
 '5.',
 '6',
 '6',
 '5.',
 '6',
 '7.',
 '7.',
 '6',
 '1.',
 '6.',
 '5.',
 '4.',
 '7.',
 '5.',
 '6.',
 '4.',
 '8.',
 '5.',
 '8.',
 '5.',
 '6',
 '6.',
 '4',
 '1',
 '5.',
 '7',
 '7',
 '7',
 '1.',
 '5',
 '8.',
 '6.',
 '7.',
 '6.',
 '6',
 '6',
 '6',
 '5.',
 '8.',
 '8.',
 '1',
 '7.',
 '6',
 '6',
 '7.',
 '5.',
 '3',
 '6',
 '6.',
 '6.',
 '6',
 '4.',
 '7.',
 '5.',
 '6.',
 '6',
 '6.',
 '8.',
 '6.',
 '6',
 '5.',
 '5.',
 '8.',
 '3',
 '6',
 '9.',
 '6',
 '3.',
 '6',
 '8',
 '7.',
 '8.',
 '8.',
 '6',
 '6.',
 '6',
 '6',
 '5',
 '3.',
 '8.',
 '8.',
 '6',
 '4',
 '9.',
 '6',
 '6.',
 '8.',
 '7.',
 '6.',
 '6',
 '6',
 '7',
 '6.',
 '6',
 '7.',
 '8',
 '6.',
 '5.',
 '6.',
 '6.',
 '5.',
 '7.',
 '6',
 '9.',
 '4',
 '6.',
 '5',
 '6.',
 '6',
 '5.',
 '5.',
 '5.',
 '9.',
 '7.',
 '6',
 '8.',
 '9',
 '6.',
 '8.',
 '5.',
 '6',
 '8.'

In [18]:
len(responses)

250

In [10]:
len(actuals)

250

In [24]:
df = pd.read_csv('classification_NFFA_llama.csv')

In [27]:
df = df.drop('Unnamed: 0', axis=1)

In [42]:
actuals = df['actuals'].to_list()
responses = df['raw_predictions'].to_list()

In [43]:
import re

predictions = []
for idx, path in enumerate(all_files):
    numbers = re.findall(r'\d+', responses[idx])
    pred = -1
    if len(numbers) > 0:
        pred = int(numbers[0])
    predictions.append(pred)

In [44]:
predictions

[8,
 5,
 3,
 6,
 6,
 9,
 6,
 9,
 9,
 6,
 3,
 5,
 3,
 5,
 1,
 3,
 6,
 5,
 9,
 3,
 6,
 6,
 3,
 6,
 6,
 5,
 3,
 6,
 3,
 6,
 9,
 6,
 3,
 6,
 5,
 6,
 6,
 5,
 4,
 6,
 1,
 1,
 6,
 5,
 3,
 5,
 6,
 5,
 6,
 9,
 3,
 8,
 6,
 1,
 8,
 5,
 3,
 6,
 1,
 9,
 9,
 6,
 3,
 7,
 5,
 5,
 8,
 5,
 6,
 6,
 9,
 6,
 1,
 1,
 5,
 5,
 8,
 5,
 7,
 6,
 5,
 8,
 3,
 7,
 5,
 6,
 5,
 6,
 6,
 8,
 5,
 1,
 5,
 3,
 3,
 9,
 5,
 8,
 5,
 6,
 6,
 5,
 6,
 6,
 7,
 3,
 5,
 6,
 7,
 3,
 6,
 6,
 3,
 5,
 8,
 6,
 6,
 6,
 1,
 8,
 6,
 8,
 3,
 7,
 3,
 9,
 5,
 6,
 6,
 3,
 5,
 6,
 5,
 5,
 3,
 8,
 5,
 5,
 1,
 5,
 1,
 6,
 9,
 1,
 7,
 6,
 8,
 6,
 3,
 8,
 5,
 6,
 5,
 6,
 5,
 7,
 6,
 6,
 1,
 6,
 6,
 6,
 5,
 8,
 3,
 1,
 6,
 6,
 5,
 6,
 5,
 8,
 7,
 5,
 8,
 6,
 5,
 7,
 5,
 6,
 7,
 5,
 5,
 5,
 7,
 5,
 3,
 5,
 5,
 3,
 5,
 6,
 6,
 5,
 7,
 5,
 1,
 8,
 3,
 7,
 7,
 7,
 3,
 6,
 5,
 5,
 6,
 1,
 5,
 9,
 6,
 8,
 5,
 8,
 8,
 6,
 8,
 8,
 5,
 5,
 7,
 5,
 5,
 3,
 5,
 7,
 5,
 7,
 7,
 7,
 1,
 8,
 6,
 7,
 1,
 9,
 5,
 8,
 5,
 6,
 7,
 7,
 6,
 9,
 5,
 6,
 5,
 5,
 6,
 5,


In [45]:
import pandas as pd
df = pd.DataFrame({'img_path': all_files, 'raw_predictions': responses, 'actuals': actuals, 'predictions': predictions})

In [46]:
len(all_files)

21169

In [47]:
len(responses)

21169

In [48]:
len(actuals)

21169

In [49]:
len(predictions)

21169

In [50]:
df.to_csv('classification_NFFA_llama.csv')